# APS360 — Transcript Classifier: Colab Training

> **Before running:** Enable GPU via `Runtime → Change runtime type → T4 GPU`

## 1. Clone repo & install dependencies

In [ ]:
import os

REPO_URL = "https://github.com/romeluis/APS360-TranscriptClassifier.git"
REPO_DIR = "/content/APS360-TranscriptClassifier"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print("Repo already cloned — pulled latest changes.")

%cd {REPO_DIR}

In [ ]:
# Python packages
!pip install -q -r requirements.txt

# WeasyPrint system libs — required to render HTML → PDF for training data
!apt-get install -q -y libpango-1.0-0 libpangoft2-1.0-0 libharfbuzz0b 2>/dev/null

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nDevice: {device}")
if device == "cuda":
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU found — training will be very slow!")

import multiprocessing
print(f"CPUs available: {multiprocessing.cpu_count()}")

## 2. Generate training data

Renders each HTML template → PDF → extracts text with pypdf → aligns entity labels.
This trains the model on **the same noisy PDF text it will see at inference time**.

- `--pdf-extract`: use PDF round-trip (HTML → WeasyPrint → PDF → pypdf text) for labels
- `--augment-noise`: add token-level noise for extra robustness
- `--no-pdf`: delete rendered PDFs after extraction (save disk space)
- `--workers 2`: matches Colab's 2 CPU cores (faster than 4 workers due to no contention)

**~15–20 min with 2 workers. Skip if data already exists.**

In [ ]:
import glob
existing = glob.glob("transcript_generator/output/*/*.json")
print(f"Existing JSON files: {len(existing)}")

# 54 templates × 50 transcripts = 2700 total
EXPECTED = 54 * 50
if len(existing) < EXPECTED:
    print(f"Generating data ({EXPECTED} transcripts expected)...")
    !python transcript_generator/main.py \
        --all-templates \
        --count 50 \
        --pdf-extract \
        --augment-noise \
        --no-pdf \
        --workers 2
else:
    print("Data already present — skipping generation.")

## 3. Train

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from model.train import train

history = train(
    data_dir="transcript_generator/output",
    output_dir="model/checkpoints",
    use_fp16=True,   # FP16 is supported on Colab T4
)

## 4. Plot training curves

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(epochs, history["train_loss"], "o-", label="train")
axes[0].plot(epochs, history["val_loss"],   "s-", label="val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history["val_token_acc"], "o-", color="royalblue")
axes[1].set_title("Val Token Accuracy"); axes[1].grid(alpha=0.3)

axes[2].plot(epochs, history["val_entity_f1"], "o-", color="green")
axes[2].set_title("Val Entity F1"); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_history.png", dpi=150)
plt.show()

## 5. Save checkpoint to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/APS360_checkpoints"
!mkdir -p {SAVE_DIR}
!cp -r model/checkpoints/best     {SAVE_DIR}/
!cp model/checkpoints/split_info.json {SAVE_DIR}/ 2>/dev/null || true
!cp model/logs/history.json        {SAVE_DIR}/ 2>/dev/null || true
!cp training_history.png           {SAVE_DIR}/ 2>/dev/null || true

print(f"Saved to Google Drive: {SAVE_DIR}")

## 6. (Optional) Download checkpoint directly

Alternative to Drive — zip and download straight to your Mac.

In [ ]:
!zip -r bert_checkpoint.zip model/checkpoints/best model/logs/history.json training_history.png

from google.colab import files
files.download("bert_checkpoint.zip")

## 7. (Optional) Quick smoke test on a test-split sample

In [ ]:
import json, glob
from model.predict import BertNERPredictor
from evaluation.reconstructor import reconstruct_semesters

with open("model/checkpoints/split_info.json") as f:
    split_info = json.load(f)

test_template = split_info["test_templates"][0]
sample_path = sorted(glob.glob(f"transcript_generator/output/{test_template}/*.json"))[0]
with open(sample_path) as f:
    sample = json.load(f)

predictor = BertNERPredictor(checkpoint_dir="model/checkpoints/best")
tags = predictor.predict(sample["tokens"])
semesters = reconstruct_semesters(sample["tokens"], tags)

print(f"Template: {test_template}\n")
for sem in semesters:
    print(f"  {sem['semester_name']}")
    for c in sem["courses"]:
        print(f"    {c['code']:<12} {c['name']:<40} {c['grade']}")